# ✈️ Extracción y Procesamiento Inicial de Datos de Tráfico Aéreo (OpenSky)

La disponibilidad de datos aeronáuticos en tiempo real permite analizar patrones de movilidad aérea, comportamiento de aeronaves y variaciones operativas a escala global.  
Este proyecto desarrolla un **pipeline ETL** que ingesta información del endpoint público de **OpenSky Network**, centrado en capturar el estado actual de miles de vuelos activos en simultáneo.

La API proporciona variables clave como:

- identificador **ICAO24**  
- país de origen  
- latitud / longitud  
- altitudes barométrica y geométrica  
- velocidad, rumbo, tasa vertical  
- estado en tierra o en vuelo  
- timestamp del servidor  

Estos datos se transforman y almacenan en un **Data Lake local** siguiendo la arquitectura **Bronze → Silver → Gold**, lo que permite realizar análisis históricos, construir métricas aeronáuticas y preparar la futura migración a entornos de nube (Azure).

## Objetivos

**Extracción (Bronze):**
- Consumir el endpoint `states/all` de OpenSky.  
- Normalizar la estructura JSON y convertirla en tabla.  
- Incorporar timestamps (servidor y extracción).  
- Guardar los datos crudos en **Delta Lake**.

**Transformación (Silver):**
- Limpiar valores faltantes y tipos de datos.  
- Estandarizar columnas y coordenadas.  
- Preparar la tabla para análisis temporal.

**Métricas (Gold):**
- Crear features de movilidad aérea: altitud efectiva, variación de velocidad, indicadores de vuelo/estacionamiento, etc.  
- Generar datasets optimizados para visualización y análisis exploratorio.

## Alcance y supuestos

- Se utilizan exclusivamente datos públicos provistos por OpenSky Network.  
- La extracción se realiza bajo límites de la API pública (sin autenticación obligatoria).  
- El objetivo es **práctico y educativo**, orientado al portfolio de Ingeniería de Datos.

## Reproducibilidad

- Dependencias detalladas en `requirements.txt`.  
- Las rutas del Data Lake se configuran en `pipeline.conf`.  
- Todas las funciones auxiliares se encuentran en `src/etl_utils.py`.

---

**Estructura del notebook:**

0) Configuración inicial  
1) Extracción del endpoint `states/all`  
2) Normalización del JSON  
3) Limpieza y estandarización mínima  
4) Almacenamiento en Delta Lake (capa Bronze)  
5) Verificación y vista preliminar de los datos


## 0. Configuración inicial

En este paso se importan todas las librerías necesarias y las funciones auxiliares definidas en `etl_utils.py`.  
Este enfoque permite mantener el notebook **ordenado, modular y fácilmente reproducible**, centralizando en un único módulo las operaciones comunes del pipeline ETL: extracción desde la API pública de **OpenSky Network**, normalización del JSON, estandarización de columnas y escritura en las distintas capas del **Data Lake local** (Bronze → Silver → Gold).

El objetivo de esta sección es garantizar que todas las dependencias estén correctamente cargadas antes de iniciar el proceso de extracción y almacenamiento.

In [1]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

# Importar funciones auxiliares
from etl_utils import *

# Librerías comunes
import pandas as pd

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


## 1. Autenticación y lectura de configuración

La configuración del proyecto se administra mediante el archivo `pipeline.conf`,  
que centraliza parámetros como:
- la **URL base** de la API de OpenSky Network  
- credenciales opcionales para *Basic Auth* (en caso de usarse)  
- rutas del **Data Lake local**

Aunque la API pública de OpenSky no requiere autenticación obligatoria,almacenar parámetros en un archivo de configuración permite:
- mantener el notebook limpio  
- evitar credenciales expuestas en el código  
- facilitar la migración futura a servicios en la nube (Azure Key Vault)

El archivo se lee mediante `ConfigParser`, lo que permite obtener los valores  
en forma segura y reusable.


In [2]:
# Se instancia el parser y se lee el archivo de configuración
from configparser import ConfigParser

parser = ConfigParser()
parser.read("../pipeline.conf")

['../pipeline.conf']

In [3]:
# Parámetros de conexión
api_config = parser["api-opensky"]
base_url = api_config["base_url"]

In [4]:
print("📄 Configuración cargada correctamente.")
print(f"URL base: {base_url}")

📄 Configuración cargada correctamente.
URL base: https://opensky-network.org/api/states/all


In [5]:
# Prueba de conexión a la API OpenSky
response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    print(f"La petición fue exitosa. Tipo de respuesta: {type(data)}")

    # Claves principales del JSON
    print(f"Claves principales recibidas: {list(data.keys())[:5]}")

    # Inspección parcial de 'states'
    print("\nPrimeras 2 aeronaves registradas:")
    pprint(data["states"][:2])

else:
    print(f"❌ Error en la petición: {response.status_code}, {response.content}")
print("✅ Prueba de conexión a la API realizada.")

La petición fue exitosa. Tipo de respuesta: <class 'dict'>
Claves principales recibidas: ['time', 'states']

Primeras 2 aeronaves registradas:
[['39de4f',
  'TVF8939 ',
  'France',
  1763546936,
  1763546937,
  8.8011,
  44.2954,
  11582.4,
  False,
  206.46,
  326.07,
  -0.33,
  None,
  11567.16,
  '6176',
  False,
  0],
 ['4b1815',
  'SWR96M  ',
  'Switzerland',
  1763546937,
  1763546937,
  4.9083,
  42.4873,
  10972.8,
  False,
  258.22,
  42.17,
  0.33,
  None,
  11026.14,
  '1000',
  False,
  0]]
✅ Prueba de conexión a la API realizada.


## 2. Capa Bronze — Extracción y almacenamiento de datos crudos

En esta etapa se realiza la **extracción directa de datos** desde la API pública de **OpenSky Network**, que provee información en tiempo real sobre aeronaves detectadas a nivel global.  
El objetivo es obtener el conjunto completo de registros tal como es devuelto por el endpoint `states/all` y conservarlo en su forma más fiel dentro de la capa **🟤 Bronze** del Data Lake.

Los datos obtenidos incluyen:

- Identificador único **ICAO24**  
- Indicativo de llamada (**callsign**)  
- País de origen  
- Posición geográfica (latitud, longitud)  
- Altitudes barométrica y geométrica  
- Velocidad, rumbo y tasa vertical  
- Estado operativo (en tierra o en vuelo)  
- Timestamp del servidor (`time`) correspondiente a la captura

Cada aeronave es representada inicialmente como una lista ordenada de 17 elementos, por lo que en esta etapa se prioriza **preservar los datos crudos** antes de aplicar procesos de estructuración o limpieza.

Los datos se almacenan en **formato Delta Lake**, dentro del directorio:

`data/etl_datalake/bronze/api_opensky/`

empleando el modo **`overwrite`**, ya que la información corresponde a un snapshot puntual del estado global del tráfico aéreo y puede reemplazarse completamente en cada actualización.


### 2.1 Extracción de datos estáticos

Se realiza una *ingesta full* sobre el recurso estático `aircraftDatabase.csv` provisto por **OpenSky Network**, que contiene información descriptiva sobre aeronaves (modelo, fabricante, typecode, operador, entre otros metadatos relevantes).

Si bien este dataset no proviene de un endpoint API, constituye la **fuente oficial de referencia** publicada por la plataforma.  
Dado que cambia muy poco en el tiempo, se incorpora en su totalidad en cada ejecución y se sobrescribe la versión previa (*mode="overwrite"*).

Este enfoque evita duplicados, simplifica el pipeline y garantiza la disponibilidad de la versión más reciente.

Los datos se almacenan en la capa 🟤 **Bronze** en formato **Delta Lake**, preservados tal como fueron obtenidos.

In [6]:
# --- Extracción de datos estáticos — OpenSky aircraft metadata ---

# URL del recurso estático oficial
metadata_url = "https://opensky-network.org/datasets/metadata/aircraftDatabase.csv"

# Descarga del archivo CSV
# Nota: Estos metadatos cambian muy poco, por lo que se aplicará ingesta "full"
df_aircraft = pd.read_csv(metadata_url)

# Vista preliminar del dataset cargado
print("📄 Dataset de metadatos cargado correctamente.")
print(f"Filas: {df_aircraft.shape[0]} | Columnas:, {df_aircraft.shape[1]}")
df_aircraft.head(3)

📄 Dataset de metadatos cargado correctamente.
Filas: 520000 | Columnas:, 27


,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,NaN,L1P,NaN,...,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,NaN,L2P,NaN,...,NaN,1977-01-01,NaN,NaN,LYCOMING TI0-540 SER,False,False,False,NaN,NaN


### 2.2 Extracción de datos dinámicos

Para los datos dinámicos se utiliza la información proveniente del endpoint `states/all`, que ofrece el estado en tiempo real de miles de aeronaves a nivel global.  
Dado que estos datos se actualizan continuamente, cada ejecución captura un *snapshot* independiente del tráfico aéreo del momento.

En este caso no se aplica una actualización incremental, ya que el conjunto de aeronaves presentes varía en cada consulta y no existe un identificador temporal que permita rastrear cambios de manera estricta.  
Por ello, cada snapshot se almacena íntegramente en la capa 🟤 *Bronze*, preservando la evolución temporal entre ejecuciones y habilitando análisis posteriores en Silver y Gold.

In [7]:
# --- Extracción de datos dinámicos — OpenSky states/all ---

# Obtención del snapshot dinámico mediante la función auxiliar
json_data = get_opensky_states()

# Vista preliminar del JSON recibido
print("🔑 Claves principales:", list(json_data.keys()))
print("🛫 Cantidad de aeronaves detectadas:", len(json_data["states"]))

# Visualización de los primeros registros crudos
print("\nPrimeras 2 aeronaves (formato crudo):")
pprint(json_data["states"][:2])

🔑 Claves principales: ['time', 'states']
🛫 Cantidad de aeronaves detectadas: 5668

Primeras 2 aeronaves (formato crudo):
[['39de4f',
  'TVF8939 ',
  'France',
  1763547008,
  1763547008,
  8.6969,
  44.4061,
  11582.4,
  False,
  206.03,
  325.99,
  0,
  None,
  11567.16,
  '6176',
  False,
  0],
 ['4b1815',
  'SWR96M  ',
  'Switzerland',
  1763547009,
  1763547009,
  5.0598,
  42.6111,
  10972.8,
  False,
  258.57,
  42.26,
  0,
  None,
  11018.52,
  '1000',
  False,
  0]]


### 2.3 Guardado en Bronze — Delta Lake

Una vez realizada la extracción de los datos estáticos y dinámicos, ambos recursos se almacenan en la capa 🟤 *Bronze* del Data Lake en formato **Delta Lake**.  
En esta etapa no se aplican transformaciones ni procesos de limpieza: los datos se preservan tal como fueron obtenidos desde la API, cumpliendo el propósito de Bronze como zona de almacenamiento crudo.

Los metadatos estáticos se guardan mediante una *ingesta full* (mode="overwrite"), ya que su contenido cambia muy poco y resulta más simple reemplazar el dataset completo en cada ejecución.

Los datos dinámicos provenientes de `states/all` se guardan como snapshots independientes, manteniendo la trazabilidad temporal de cada captura.  
Este enfoque permite conservar el historial de estados del tráfico aéreo para posteriores análisis en las capas Silver y Gold.

Las rutas de salida se definen dentro del directorio:

`data/etl_datalake/bronze/`

utilizando subdirectorios separados para **datos estáticos** y **datos dinámicos**.

In [8]:
# --- Definición de rutas del Data Lake (Bronze) ---

# Se mantiene fuera de la carpeta notebooks para centralizar los datos.
datalake_root = "../data/etl_datalake"

# Carpeta raíz del dominio OpenSky dentro de Bronze
bronze_dir = f"{datalake_root}/bronze/api_opensky"

# Subcarpetas descriptivas según el tipo de recurso
static_dir  = f"{bronze_dir}/aircraft_metadata"
dynamic_dir = f"{bronze_dir}/states"

print("📁 Rutas definidas:")
print("Static  →", static_dir)
print("Dynamic →", dynamic_dir)

📁 Rutas definidas:
Static  → ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
Dynamic → ../data/etl_datalake/bronze/api_opensky/states


In [9]:
# --- Guardado en Bronze — Metadatos estáticos (Delta Lake) ---

# Ingesta full: se sobrescribe el dataset completo en cada ejecución,
# dado que los metadatos de aeronaves cambian muy poco en el tiempo.
save_data_as_delta(
    df=df_aircraft,
    path=static_dir,
    mode="overwrite"
)

print("🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).")

💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/aircraft_metadata
🟤 Datos estáticos guardados correctamente en Bronze (Delta Lake).


In [10]:
# --- Guardado en Bronze — Snapshot dinámico (Delta Lake) ---

# Conversión a DataFrame del recurso dinámico "states"
df_dynamic = pd.DataFrame(json_data["states"])

# En algunos snapshots, ciertas columnas pueden venir completamente vacías.
# Delta Lake no acepta columnas de tipo "Null" (100% None), por lo que se eliminan
# únicamente aquellas que no contienen ningún valor real.
df_dynamic = df_dynamic.dropna(axis=1, how="all")

print("📄 Shape del snapshot después de eliminar columnas vacías:", df_dynamic.shape)

# Guardado incremental (append) para preservar el historial de snapshots.
save_data_as_delta(
    df=df_dynamic,
    path=dynamic_dir,
    mode="append"
)

print("🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).")

📄 Shape del snapshot después de eliminar columnas vacías: (5668, 16)
💾 Datos guardados en Delta Lake: ../data/etl_datalake/bronze/api_opensky/states
🟤 Snapshot dinámico guardado correctamente en Bronze (Delta Lake).


## 3. Capa Silver — Normalización y limpieza

La capa **Silver** aplica transformaciones sobre los datos almacenados en Bronze con el objetivo de obtener un conjunto de datos limpio, tabular y estructurado, adecuado para análisis posteriores.

En esta etapa se realizan:
- renombrado de columnas  
- estandarización de tipos  
- conversión de timestamps  
- selección de atributos relevantes  
- reducción de nulos  
- enriquecimiento opcional mediante cruce entre datasets

El procesamiento se organiza en dos pasos:

### 3.1 Metadatos estáticos (aircraft_metadata)
Depuración de columnas, estandarización de nombres y construcción de la **tabla de referencia de aeronaves**, que funcionará como conjunto de metadatos estable para enriquecer los datos dinámicos.

### 3.2 Datos dinámicos (states/all)
Normalización del snapshot, asignación de nombres descriptivos a las columnas, tipificación y preparación para análisis temporal.

Este enfoque garantiza un esquema consistente y una base sólida para la etapa **Gold**, donde se generarán métricas, indicadores y visualizaciones del tráfico aéreo.

### 3.1 Metadatos estáticos — Construcción de la tabla de referencia de aeronaves

El dataset de metadatos de aeronaves obtenido desde OpenSky contiene información descriptiva asociada a cada código `icao24`.  
Aunque estos datos cambian poco en el tiempo, llegan con numerosas columnas incompletas, valores inconsistentes y atributos que no aportan información útil.

En esta etapa se construye una **tabla de referencia de aeronaves** limpia y estable mediante los siguientes pasos:

- **eliminación de columnas con nulos al 100%**
- **verificación y eliminación de duplicados** en la clave primaria `icao24`
- **detección y eliminación de columnas sin variabilidad** (valores constantes)
- **selección de atributos relevantes**, conservando únicamente información descriptiva útil:
  - `icao24`
  - `registration`
  - `manufacturername`
  - `model`
  - `typecode`
  - `owner`
  - `built`
  - `engines`
- **tipificación suave**, asegurando que todos los campos descriptivos estén representados como `string`

El resultado es una tabla compacta, consistente y adecuada para enriquecer los datos dinámicos procesados en la siguiente etapa.

In [11]:
# Se lee Bronze (Delta)
dt_aircraft_metadata = DeltaTable(static_dir)

In [12]:
# Se convierte a Pandas para trabajar en las transformaciones
dt_aircraft_metadata_bronze = dt_aircraft_metadata.to_pandas()

In [13]:
# Se crea una copia para aplicar las transformaciones
df_aircraft_cleaned  = dt_aircraft_metadata_bronze.copy()

In [14]:
# Vista preliminar
df_aircraft_cleaned.head(3)

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
0,None,None,None,None,None,None,None,None,None,None,...,NaN,None,None,NaN,None,False,False,False,None,None
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,None,L1P,None,...,NaN,None,None,NaN,None,False,False,False,None,None
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,None,L2P,None,...,NaN,1977-01-01,None,NaN,LYCOMING TI0-540 SER,False,False,False,None,None


In [15]:
# --- Inspección inicial del dataset estático 
df_aircraft_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520000 entries, 0 to 519999
Data columns (total 27 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   icao24               519999 non-null  object 
 1   registration         516526 non-null  object 
 2   manufacturericao     428801 non-null  object 
 3   manufacturername     438566 non-null  object 
 4   model                440426 non-null  object 
 5   typecode             479947 non-null  object 
 6   serialnumber         437118 non-null  object 
 7   linenumber           971 non-null     object 
 8   icaoaircrafttype     428784 non-null  object 
 9   operator             23698 non-null   object 
 10  operatorcallsign     40090 non-null   object 
 11  operatoricao         41402 non-null   object 
 12  operatoriata         8066 non-null    object 
 13  owner                435913 non-null  object 
 14  testreg              297 non-null     object 
 15  registered       

In [16]:
# --- Verificación de valores nulos en la clave primaria (icao24) ---
nulos = df_aircraft_cleaned['icao24'].isna().sum()
print(f"Cantidad de valores nulos en 'icao24': {nulos}")

Cantidad de valores nulos en 'icao24': 1


In [17]:
# --- Verificación de valores vacíos o espacios en 'icao24' ---
vacios = (df_aircraft_cleaned['icao24'].astype(str).str.strip() == "").sum()
print(f"Cantidad de valores vacíos en 'icao24': {vacios}")

Cantidad de valores vacíos en 'icao24': 0


In [18]:
# --- Verificación de duplicados en 'icao24' ---
duplicados = df_aircraft_cleaned['icao24'].duplicated().sum()
print(f"Cantidad de valores duplicados en 'icao24': {duplicados}")

Cantidad de valores duplicados en 'icao24': 2


In [19]:
# --- Inspección de los valores duplicados en 'icao24' ---

# Filtramos únicamente las filas cuyo icao24 aparece más de una vez
df_aircraft_cleaned[df_aircraft_cleaned['icao24'].duplicated(keep=False)].sort_values('icao24')

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categoryDescription
31226,ae690b,710376,PANHA,None,None,B06,None,None,H1T,None,...,NaN,None,None,NaN,None,False,False,False,None,None
423031,ae690b,710376,PANHA,None,None,B06,None,None,H1T,None,...,NaN,None,None,NaN,None,False,False,False,None,None
82947,ae6963,170000,LOCKHEED MARTIN,Lockheed,C-130J Hercules C.5,C30J,5483,None,L4T,United States Navy,...,NaN,None,None,NaN,None,False,False,False,"2020 July: ""Blue Angels"" Team Support Aircaft,...",Reserved
302446,ae6963,170000,LOCKHEED MARTIN,Lockheed,C-130J Hercules C.5,C30J,5483,None,L4T,United States Navy,...,NaN,None,None,NaN,None,False,False,False,"2020 July: ""Blue Angels"" Team Support Aircaft,...",Reserved


In [20]:
# --- Eliminación de valores nulos en la clave primaria (icao24) ---
df_aircraft_cleaned = df_aircraft_cleaned[df_aircraft_cleaned['icao24'].notna()]

In [21]:
# --- Eliminación de duplicados en la clave primaria (icao24) ---
df_aircraft_cleaned = df_aircraft_cleaned.drop_duplicates(subset='icao24', keep='first')

In [22]:
# --- Verificación post-limpieza ---
print("Nulos en icao24:", df_aircraft_cleaned['icao24'].isna().sum())
print("Duplicados en icao24:", df_aircraft_cleaned['icao24'].duplicated().sum())

Nulos en icao24: 0
Duplicados en icao24: 0


In [23]:
    # --- Estandarización de nombres de columnas (Silver) ---
# Se normalizan todos los nombres de columnas para garantizar un esquema consistente.
# Este formato (snake_case + minúsculas) es el estándar utilizado en pipelines ETL
# y evita ambigüedades en transformaciones posteriores y uniones entre datasets.

df_aircraft_cleaned.columns = (
    df_aircraft_cleaned.columns
        .str.lower()            # forzar minúsculas
        .str.strip()            # eliminar espacios al inicio y final
        .str.replace(r"\s+", "_", regex=True)   # reemplazar espacios por '_'
        .str.replace(r"[^a-z0-9_]", "", regex=True)  # eliminar caracteres no válidos
)
df_aircraft_cleaned.head(3)

,icao24,registration,manufacturericao,manufacturername,model,typecode,serialnumber,linenumber,icaoaircrafttype,operator,...,status,built,firstflightdate,seatconfiguration,engines,modes,adsb,acars,notes,categorydescription
1,aa3487,N757F,RAYTHEON,Raytheon Aircraft Company,A36,BE36,E-3121,None,L1P,None,...,NaN,None,None,NaN,None,False,False,False,None,None
2,a4fa61,N42MH,PIPER,Piper,PA-31-350,PA31,31-7852015,None,L2P,None,...,NaN,1977-01-01,None,NaN,LYCOMING TI0-540 SER,False,False,False,None,None
3,a7a809,N5926K,ROCKWELL,None,None,AC90,None,None,L2T,None,...,NaN,None,None,NaN,None,False,False,False,None,None


In [24]:
# --- Eliminación de columnas con 100% de valores nulos ---
# En esta etapa se descartan las columnas cuyo contenido es completamente nulo.
# Estas variables no aportan información a la tabla de referencia de aeronaves y
# solo generarían ruido en las siguientes etapas (joins, estadísticas, almacenamiento).

null_pct = df_aircraft_cleaned.isna().mean() * 100
cols_to_drop = null_pct[null_pct == 100].index.tolist()

df_aircraft_cleaned = df_aircraft_cleaned.drop(columns=cols_to_drop)

print(f"📉 Columnas eliminadas por nulos al 100%: {cols_to_drop}")
print(f"📦 Total de columnas restantes: {df_aircraft_cleaned.shape[1]}")

📉 Columnas eliminadas por nulos al 100%: ['status', 'seatconfiguration']
📦 Total de columnas restantes: 25


In [25]:
# --- Definición de columnas candidatas según documentación de OpenSky ---

# Atributos principales esperados en la tabla de referencia de aeronaves
cols_principales = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "operator"
]

# Atributos complementarios (útiles si contienen variabilidad)
cols_complementarias = [
    "owner",
    "built",
    "engines",
    "modes",
    "adsb",
    "acars"
]

# Combinación de columnas candidatas
cols_candidatas = cols_principales + cols_complementarias

# Se filtran solo las columnas presentes en el dataset
cols_candidatas = [c for c in cols_candidatas if c in df_aircraft_cleaned.columns]

print("📌 Columnas candidatas detectadas:", cols_candidatas)


📌 Columnas candidatas detectadas: ['icao24', 'registration', 'manufacturername', 'model', 'typecode', 'operator', 'owner', 'built', 'engines', 'modes', 'adsb', 'acars']


In [26]:
# --- Verificación de variabilidad para cada columna candidata ---

print("🔎 Verificación de variabilidad en columnas candidatas:\n")

cols_constantes = []

for col in cols_candidatas:
    valores_unicos = df_aircraft_cleaned[col].dropna().unique()
    n_unicos = len(valores_unicos)

    print(f"{col}: {n_unicos} valores únicos")

    # Si la columna tiene 0 o 1 valores distintos → no aporta información
    if n_unicos <= 1:
        cols_constantes.append(col)

print("\n⚠️ Columnas con variabilidad nula:", cols_constantes)

🔎 Verificación de variabilidad en columnas candidatas:

icao24: 519997 valores únicos
registration: 514254 valores únicos
manufacturername: 41540 valores únicos
model: 34482 valores únicos
typecode: 1933 valores únicos
operator: 4001 valores únicos
owner: 224893 valores únicos
built: 381 valores únicos
engines: 7529 valores únicos
modes: 1 valores únicos
adsb: 1 valores únicos
acars: 1 valores únicos

⚠️ Columnas con variabilidad nula: ['modes', 'adsb', 'acars']


In [27]:
# --- Eliminación de columnas sin información útil ---

df_aircraft_cleaned.drop(columns=cols_constantes, inplace=True)

print("🧹 Columnas eliminadas:", cols_constantes)
print("📦 Columnas restantes:", df_aircraft_cleaned.columns.tolist())

🧹 Columnas eliminadas: ['modes', 'adsb', 'acars']
📦 Columnas restantes: ['icao24', 'registration', 'manufacturericao', 'manufacturername', 'model', 'typecode', 'serialnumber', 'linenumber', 'icaoaircrafttype', 'operator', 'operatorcallsign', 'operatoricao', 'operatoriata', 'owner', 'testreg', 'registered', 'reguntil', 'built', 'firstflightdate', 'engines', 'notes', 'categorydescription']


In [28]:
# --- Selección final de atributos relevantes para la tabla de referencia de aeronaves ---

cols_finales = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "owner",
    "built",
    "engines"
]

# Verificación final: se mantienen solo las columnas presentes tras la depuración
cols_finales = [c for c in cols_finales if c in df_aircraft_cleaned.columns]

# Construcción de la tabla de referencia (versión Silver)
df_aircraft_cleaned = df_aircraft_cleaned[cols_finales]

print("📘 Tabla de referencia de aeronaves — Silver:")
df_aircraft_cleaned.head(5)

📘 Tabla de referencia de aeronaves — Silver:


,icao24,registration,manufacturername,model,typecode,owner,built,engines
1,aa3487,N757F,Raytheon Aircraft Company,A36,BE36,Vintage Aircraft Llc,None,None
2,a4fa61,N42MH,Piper,PA-31-350,PA31,Tvpx Aircraft Solutions Inc Trustee,1977-01-01,LYCOMING TI0-540 SER
3,a7a809,N5926K,None,None,AC90,None,None,None
4,391927,F-GGJH,Robin,DR.400 160 Chevalier,DR40,Private,None,None
5,503c21,LY-KNA,Impulse Aircraft,Impulse 100,ZZZZ,Private,None,None


In [29]:
# --- Verificación inicial de tipos de datos ---
# Se revisa el tipo actual de cada columna antes de la tipificación suave.
df_aircraft_cleaned.dtypes

icao24              object
registration        object
manufacturername    object
model               object
typecode            object
owner               object
built               object
engines             object
dtype: object

In [ ]:
# --- Tipificación suave: se asegura tipo string en columnas descriptivas ---
# Convertimos columnas 'object' a string para evitar valores mixtos 
# (mezclas de números, nulos y cadenas que suelen aparecer en datos provenientes de APIs).
# Se transforman explícitamente a dtype "string" para garantizar consistencia en Silver.
# Esta explicación detallada se incluye solo en 3.1; en 3.2 la conversión se aplicará de forma más breve.

cols_string = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "owner",
    "built",
    "engines"
]

# Conversión a string (astype evita problemas con valores mixtos)
df_aircraft_cleaned[cols_string] = df_aircraft_cleaned[cols_string].astype("string")

print("✅ Tipificación suave aplicada (todas las columnas relevantes como string).")
df_aircraft_cleaned.dtypes

cols_string = [
    "icao24",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "owner",
    "built",
    "engines"
]

# Conversión a string (astype evita problemas con valores mixtos)
df_aircraft_cleaned[cols_string] = df_aircraft_cleaned[cols_string].astype("string")

print("✅ Tipificación suave aplicada (todas las columnas relevantes como string).")
df_aircraft_cleaned.dtypes

✅ Tipificación suave aplicada (todas las columnas relevantes como string).
✅ Tipificación suave aplicada (todas las columnas relevantes como string).


icao24              string[python]
registration        string[python]
manufacturername    string[python]
model               string[python]
typecode            string[python]
owner               string[python]
built               string[python]
engines             string[python]
dtype: object

### 3.2 Datos dinámicos — Normalización del snapshot de estados (states/all)

El endpoint `states/all` de OpenSky proporciona un **snapshot en tiempo real** del estado de todas las aeronaves detectadas en el momento de la consulta.  
A diferencia de los metadatos estáticos, estos datos son **altamente volátiles** y cambian de un segundo a otro.

Contienen información fundamental para el análisis del tráfico aéreo:

- posición (`longitude`, `latitude`)
- altitud (`baro_altitude`)
- velocidad (`velocity`)
- rumbo (`true_track`)
- tasa de ascenso/descenso (`vertical_rate`)
- indicadores del transpondedor y estado en superficie
- timestamps (`time_position`, `last_contact`)

Como este snapshot llega como **listas anónimas**, donde cada registro es un arreglo indexado (`0`–`16`), la primera tarea es convertirlo en un **DataFrame tabular** con nombres claros.

En esta etapa se aplican las siguientes transformaciones:

- asignación de nombres descriptivos a las columnas  
- tipificación de datos numéricos, booleanos y categóricos  
- conversión de timestamps a formato `datetime`  
- eliminación de columnas sin variabilidad  
- creación de columnas temporales derivadas (`snapshot_date`, `snapshot_hour`) para permitir **particionamiento temporal** en Delta Lake  

El objetivo es obtener un dataset dinámico **limpio, tipado y consistente**, listo para su almacenamiento en Silver con esquema estable y particionado temporal.

### 3.2.1 Lectura del último snapshot dinámico desde Bronze

Los datos dinámicos del endpoint `states/all` se almacenan en Bronze como **snapshots independientes**, cada uno correspondiente al momento exacto en que fue ejecutada la ingesta.

Para iniciar el procesamiento Silver, se carga **el snapshot más reciente**, que representa el estado actual del tráfico aéreo en el momento de ejecución.  
Este enfoque permite trabajar siempre con la versión más actualizada sin perder el historial completo almacenado en Bronze.

La lectura se realiza directamente desde la tabla Delta Lake ubicada en:

`data/etl_datalake/bronze/api_opensky/states/`

Una vez cargado el dataset, se convierte a **Pandas** para aplicar las transformaciones de limpieza y normalización necesarias para la etapa Silver.

In [31]:
# --- Lectura del snapshot más reciente desde Bronze (Delta Lake) ---
dt_states = DeltaTable(dynamic_dir)

In [32]:
# Se convierte a Pandas para trabajar en las transformaciones
df_states_bronze = dt_states.to_pandas()

In [33]:
# Se crea una copia para aplicar las transformaciones
df_states_cleaned  = df_states_bronze.copy()

In [34]:
# Vista preliminar
df_states_cleaned.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,13,14,15,16
0,39de4f,TVF8939,France,1.763547e+09,1763547008,8.6969,44.4061,11582.4,False,206.03,325.99,0.0,11567.16,6176,False,0
1,4b1815,SWR96M,Switzerland,1.763547e+09,1763547009,5.0598,42.6111,10972.8,False,258.57,42.26,0.0,11018.52,1000,False,0
2,4b1817,SWR5KY,Switzerland,1.763547e+09,1763547008,4.9540,48.3432,9448.8,False,253.17,124.82,0.0,9265.92,1000,False,0


### 3.2.2 Normalización del snapshot dinámico

El endpoint `states/all` devuelve cada aeronave como una lista de 17 elementos, donde cada posición tiene un significado definido por OpenSky.  
Al cargar estos datos en Pandas, las columnas aparecen numeradas (`0`, `1`, `2`, …), por lo que el primer paso es renombrarlas utilizando los nombres oficiales.

Luego del renombrado, se aplican las transformaciones esenciales:

- **eliminación de columnas sin variabilidad** (p. ej., `position_source` con un único valor)  
- **conversión de timestamps** (`time_position`, `last_contact`) utilizando `pd.to_datetime(..., unit="s")`  
- **tipificación suave** de columnas descriptivas (`icao24`, `callsign`, `origin_country`, `squawk`)  
- mantenimiento y verificación de tipos numéricos y booleanos  
- **creación de la columna `snapshot_time`**, a partir del timestamp original del snapshot, para permitir el **particionamiento por fecha y hora** en Delta Lake  

El resultado es un snapshot dinámico **limpio, tabular y preparado para análisis temporal**, compatible con el particionado en Silver y la integración con los metadatos estáticos en etapas posteriores.

In [35]:
# --- Renombrado de columnas del snapshot dinámico (states/all) ---

column_names = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source"
]

df_states_cleaned.columns = column_names

print("✅ Columnas renombradas correctamente.")
df_states_cleaned.head(3)

✅ Columnas renombradas correctamente.


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source
0,39de4f,TVF8939,France,1.763547e+09,1763547008,8.6969,44.4061,11582.4,False,206.03,325.99,0.0,11567.16,6176,False,0
1,4b1815,SWR96M,Switzerland,1.763547e+09,1763547009,5.0598,42.6111,10972.8,False,258.57,42.26,0.0,11018.52,1000,False,0
2,4b1817,SWR5KY,Switzerland,1.763547e+09,1763547008,4.9540,48.3432,9448.8,False,253.17,124.82,0.0,9265.92,1000,False,0


In [36]:
# --- Verificación de variabilidad en columnas candidatas ---

cols_candidatas_states = ["position_source"]

cols_constantes_states = []

print("🔎 Verificación de variabilidad en columnas candidatos (states/all):\n")

for col in cols_candidatas_states:
    valores_unicos = df_states_cleaned[col].dropna().unique()
    n_unicos = len(valores_unicos)

    print(f"{col}: {n_unicos} valores únicos")

    # Si la columna tiene solo 1 valor → no aporta información
    if n_unicos <= 1:
        cols_constantes_states.append(col)

print("\n⚠️ Columnas con variabilidad nula:", cols_constantes_states)

🔎 Verificación de variabilidad en columnas candidatos (states/all):

position_source: 1 valores únicos

⚠️ Columnas con variabilidad nula: ['position_source']


In [37]:
# --- Eliminación de columnas sin información útil ---

df_states_cleaned.drop(columns=cols_constantes_states, inplace=True)

print("🧹 Columnas eliminadas (states/all):", cols_constantes_states)
print("📦 Columnas restantes:", df_states_cleaned.columns.tolist())

🧹 Columnas eliminadas (states/all): ['position_source']
📦 Columnas restantes: ['icao24', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'geo_altitude', 'squawk', 'spi']


In [38]:
# --- Verificación inicial de tipos de datos ---
print("📊 Tipos de datos antes de la normalización (Bronze → Silver):")
df_states_cleaned.dtypes

📊 Tipos de datos antes de la normalización (Bronze → Silver):


icao24             object
callsign           object
origin_country     object
time_position     float64
last_contact        int64
longitude         float64
latitude          float64
baro_altitude     float64
on_ground            bool
velocity          float64
true_track        float64
vertical_rate     float64
geo_altitude      float64
squawk             object
spi                  bool
dtype: object

In [39]:
# --- Conversión de columnas temporales a datetime ---
df_states_cleaned["time_position"] = pd.to_datetime(
    df_states_cleaned["time_position"], unit="s", errors="coerce"
)

df_states_cleaned["last_contact"] = pd.to_datetime(
    df_states_cleaned["last_contact"], unit="s", errors="coerce"
)

In [40]:
df_states_cleaned["spi"].unique()

array([False,  True])

In [41]:
df_states_cleaned["on_ground"].unique()

array([False,  True])

In [42]:
# --- Tipificación de columnas de texto a string ---
cols_string = ["icao24", "callsign", "origin_country", "squawk"]
df_states_cleaned[cols_string] = df_states_cleaned[cols_string].astype("string")

In [43]:
# --- Tipificación suave: conversión de columnas categóricas a string ---
cols_string_states = [
    "icao24",
    "callsign",
    "origin_country",
    "squawk"
]

# Conversión segura a string (evita problemas con valores mixtos)
df_states_cleaned[cols_string_states] = df_states_cleaned[cols_string_states].astype("string")

In [44]:
# --- Verificación final de tipos ---
print("📊 Tipos finales luego de la normalización Silver:")
df_states_cleaned.dtypes

📊 Tipos finales luego de la normalización Silver:


icao24            string[python]
callsign          string[python]
origin_country    string[python]
time_position     datetime64[ns]
last_contact      datetime64[ns]
longitude                float64
latitude                 float64
baro_altitude            float64
on_ground                   bool
velocity                 float64
true_track               float64
vertical_rate            float64
geo_altitude             float64
squawk            string[python]
spi                         bool
dtype: object

In [45]:
# --- Creación de columnas de timestamp para control de ingesta y particionado ---

# Se registra el momento exacto en que se procesó el snapshot.
# Esto permite identificar cada ingesta y habilita análisis temporales
# basados en la evolución del tráfico aéreo.
ingest_timestamp = datetime.utcnow()

# Timestamp completo del snapshot (precisión de segundos).
# Esta marca se mantiene uniforme para todas las filas del dataset.
df_states_cleaned["snapshot_time"] = ingest_timestamp

# Columna derivada para particionado por hora en Delta Lake.
# Se usa formato string para garantizar estabilidad de la partición
# y alineación con buenas prácticas de Data Lake (YYYY-MM-DD-HH).
df_states_cleaned["snapshot_hour"] = ingest_timestamp.strftime("%Y-%m-%d-%H")

### 3.3 Guardado en Silver — Delta Lake

Una vez finalizada la normalización tanto de los **metadatos estáticos** como del **snapshot dinámico**, ambos datasets se almacenan en la capa 🥈 *Silver* del Data Lake en formato **Delta Lake**.

En Silver se preservan los datos ya depurados y tipificados, con:

- nombres de columnas descriptivos  
- tipos corregidos (datetime, numéricos, booleanos y texto)  
- columnas sin información útil eliminadas  
- **columnas temporales derivadas** (`snapshot_time`, `snapshot_hour`) creadas para trazabilidad y particionado

El objetivo de esta capa es ofrecer una **base confiable y estructurada** para análisis temporales y enriquecimiento con otros datasets en la etapa Gold.

Los datos se almacenan en el directorio:

`data/etl_datalake/silver/api_opensky/`

con las siguientes subdivisiones:

- `aircraft_metadata/` → metadatos estáticos normalizados  
- `states/` → snapshot dinámico limpio, tipado y con columnas temporales

Para el guardado se utiliza:

- `mode="overwrite"` para metadatos estáticos (cambian muy poco en el tiempo)  
- `mode="append"` para el snapshot dinámico, **particionado por la columna `snapshot_hour`**, permitiendo conservar eficientemente el historial temporal.

In [46]:
# --- Definición de rutas del Data Lake (Silver) ---

datalake_root = "../data/etl_datalake"

silver_dir = f"{datalake_root}/silver/api_opensky"

# Subcarpetas específicas de Silver
silver_static_dir  = f"{silver_dir}/aircraft_metadata"
silver_dynamic_dir = f"{silver_dir}/states"

print("📁 Rutas Silver definidas:")
print("Static  →", silver_static_dir)
print("Dynamic →", silver_dynamic_dir)

📁 Rutas Silver definidas:
Static  → ../data/etl_datalake/silver/api_opensky/aircraft_metadata
Dynamic → ../data/etl_datalake/silver/api_opensky/states


In [47]:
# --- Guardado en Silver — Metadatos estáticos (Delta Lake) ---

save_data_as_delta(
    df=df_aircraft_cleaned,
    path=silver_static_dir,
    mode="overwrite"  # reemplaza la versión previa
)

print("🥈 Datos estáticos guardados correctamente en Silver (Delta Lake).")

💾 Datos guardados en Delta Lake: ../data/etl_datalake/silver/api_opensky/aircraft_metadata
🥈 Datos estáticos guardados correctamente en Silver (Delta Lake).


In [48]:
# --- Guardado en Silver — Snapshot dinámico particionado (Delta Lake) ---

# Se guarda el snapshot normalizado en formato Delta Lake,
# particionando por 'snapshot_hour' para habilitar almacenamiento histórico
# eficiente y consultas basadas en la evolución temporal del tráfico aéreo.
save_data_as_delta(
    df=df_states_cleaned,
    path=silver_dynamic_dir,
    mode="append",              # se acumulan snapshots sucesivos
    partition_cols=["snapshot_hour"]
)

print("🥈 Snapshot dinámico guardado correctamente en Silver (Delta Lake).")

💾 Datos guardados en Delta Lake: ../data/etl_datalake/silver/api_opensky/states
🥈 Snapshot dinámico guardado correctamente en Silver (Delta Lake).


## 4. Capa Gold — Métricas, enriquecimiento y dataset final

La capa **Gold** constituye la fase final del pipeline y tiene como objetivo producir información **lista para análisis**, integrando los datos ya normalizados en Silver y generando productos analíticos de alto nivel.

Mientras que:

- **Bronze** conserva datos crudos  
- **Silver** garantiza limpieza y consistencia  
- **Gold** entrega resultados derivados y analíticamente útiles

En esta etapa se construyen:

- métricas descriptivas del snapshot dinámico  
- indicadores operativos (velocidad, altitud, países de origen, etc.)  
- una tabla final enriquecida con metadatos estáticos  
- visualizaciones básicas (opcional)  
- un dataset consolidado listo para BI o dashboards

El objetivo no es realizar un análisis exhaustivo del tráfico aéreo, sino demostrar el cierre profesional del flujo **Bronze → Silver → Gold**, entregando datos curados y documentados.

La capa Gold se organiza en dos partes principales:

### 4.1 Métricas del snapshot dinámico  
Cálculo de estadísticas que describen el estado del tráfico aéreo capturado.

### 4.2 Enriquecimiento con la tabla de referencia de aeronaves  
Combinación del snapshot dinámico con metadatos (fabricante, modelo, typecode) para construir un dataset final analizable.


In [49]:
# --- Lectura completa desde Silver para la capa Gold ---
# Se leen todas las particiones disponibles del snapshot dinámico ya normalizado.
# En Gold trabajamos con el histórico acumulado para calcular métricas globales.
df_states_silver_all = read_all_from_delta(silver_dynamic_dir)

In [50]:
# Se crea una copia para evitar modificar la tabla original
df_states_gold = df_states_silver_all.copy()

print("Registros leídos:", df_states_gold.shape)

Registros leídos: (58325, 17)


### 4.1 Métricas del snapshot dinámico

En esta sección se generan **métricas descriptivas** basadas en el snapshot dinámico procesado en la capa Silver.  
Estas métricas permiten validar la coherencia del dataset y obtener una visión general del tráfico aéreo capturado en la ingesta.

Antes de calcular los indicadores, los datos se leen nuevamente desde Silver y se **normalizan los tipos clave**,  
ya que ciertos formatos pueden degradarse al persistir en Delta Lake (por ejemplo, columnas `string` que vuelven como `object`).  
Este paso asegura un dataset **estable, tipado y consistente** para el análisis.

#### Indicadores principales a calcular

- **Cantidad total de aeronaves detectadas**
- **Porcentaje de aeronaves en vuelo vs. en tierra**
- **Distribución de velocidades**
  - promedio  
  - mínimo y máximo  
  - percentiles relevantes  
- **Distribución de altitudes** (`baro_altitude`, `geo_altitude`)
- **Top países de origen** (`origin_country`)
- **Validación de calidad del snapshot**
  - cantidad de `callsign` válidos  
  - registros sin latitud / longitud  
  - proporción de valores nulos en velocidad o altitud  

Estas métricas cumplen un doble propósito:

1. **Evaluar el estado y la calidad del snapshot dinámico**.  
2. **Preparar el terreno para el enriquecimiento** con los metadatos de aeronaves en la sección 4.2.

In [ ]:
# --- Tipificación suave en Gold: se asegura tipo string en columnas descriptivas ---

cols_string_gold = [
    "icao24",
    "callsign",
    "origin_country",
    "squawk"
]

df_states_gold[cols_string_gold] = df_states_gold[cols_string_gold].astype("string")

print("✳️ Columnas categóricas casteadas a string:")
df_states_gold[cols_string_gold].dtypes

✳️ Columnas categóricas casteadas a string:


icao24            string[python]
callsign          string[python]
origin_country    string[python]
squawk            string[python]
dtype: object

In [58]:
# --- Métricas principales del snapshot dinámico (Gold) ---

print("📌 Métricas iniciales del snapshot dinámico\n")

# 1) Cantidad total de aeronaves
total_aircraft = len(df_states_gold)
print(f"✈️ Total de aeronaves detectadas: {total_aircraft}")

# 2) Porcentaje de aeronaves en vuelo vs en tierra
en_tierra = df_states_gold["on_ground"].sum()
en_vuelo = total_aircraft - en_tierra

pct_vuelo = (en_vuelo / total_aircraft) * 100
pct_tierra = (en_tierra / total_aircraft) * 100

print(f"🛫 En vuelo: {en_vuelo} ({pct_vuelo:.2f}%)")
print(f"🛬 En tierra: {en_tierra} ({pct_tierra:.2f}%)")

# 3) Registros sin posición geográfica
sin_posicion = df_states_gold["latitude"].isna().sum()
print(f"📍 Registros sin lat/long: {sin_posicion}")

# 4) Callsign válidos (no nulos, no vacíos)
callsign_validos = df_states_gold["callsign"].notna().sum()
print(f"🔠 Callsign válidos: {callsign_validos}")

# 5) Top 10 países de origen
top_paises = (
    df_states_gold["origin_country"]
    .value_counts()
    .head(10)
)

print("\n🌍 Top 10 países de origen:")
print(top_paises)

📌 Métricas iniciales del snapshot dinámico

✈️ Total de aeronaves detectadas: 58325
🛫 En vuelo: 53414 (91.58%)
🛬 En tierra: 4911 (8.42%)
📍 Registros sin lat/long: 483
🔠 Callsign válidos: 58325

🌍 Top 10 países de origen:
origin_country
United States     34165
Canada             2321
United Kingdom     1771
Germany            1443
Turkey             1233
Ireland            1092
Brazil              917
France              894
Spain               893
China               839
Name: count, dtype: Int64
